# © Artur Czarnecki. All rights reserved.

## Intergrax experiment template (§35)

Copy this notebook for a new capability experiment.

Workflow:

```text
1. Define hypothesis
2. Define capability + validation criteria
3. Register agent in Nexus
4. Run through NexusLoop
5. Observe trace + compare output
6. Decide: keep · improve · pause · delete
```

See also: `docs/experiment_guide.md`, `notebooks/experiments/README.md`.

### Step 0 — Repository setup

In [ ]:
from pathlib import Path

from intergrax.experiments.models import ExperimentDecision, RegisterExperimentRequest
from intergrax.experiments.workflow import ExperimentSession, ensure_repo_root_on_path
from intergrax.runtime.registry.agent_registry import AgentRegistry

REPO_ROOT = ensure_repo_root_on_path()
BUILD_DIR = REPO_ROOT / "build" / "notebooks"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print(f"repo root: {REPO_ROOT}")

### Step 1 — Define hypothesis (edit these fields)

In [ ]:
HYPOTHESIS = "<your hypothesis>"
CAPABILITY = "<capability.id>"
AGENT_ID = None  # optional explicit agent
EXPECTED_OUTPUT = ""
VALIDATION_CRITERIA = ""
TEST_MESSAGE = "hello experiment"
TENANT_ID = "t1"
USER_ID = "u1"

### Step 2 — Register experiment in registry

In [ ]:
session = ExperimentSession(
    experiments_db=BUILD_DIR / "experiments.db",
    trace_db=BUILD_DIR / "trace.db",
    tenant_id=TENANT_ID,
    user_id=USER_ID,
)

record = session.register(
    RegisterExperimentRequest(
        hypothesis=HYPOTHESIS,
        capability=CAPABILITY,
        agent_id=AGENT_ID,
        expected_output=EXPECTED_OUTPUT,
        validation_criteria=VALIDATION_CRITERIA,
    )
)
print(f"experiment_id: {record.experiment_id}")

### Step 3 — Register agent(s) and build NexusLoop

Replace the registry block with your agent module.

In [ ]:
# Example: scaffold with `python -m intergrax.scaffold new-agent my_agent`
# from my_agent.my_agent_agent import MyAgentAgent

registry = AgentRegistry()
# registry.register(MyAgentAgent())

loop = session.build_nexus_loop(registry)

### Step 4 — Run through Nexus

In [ ]:
import asyncio


async def _run():
    return await session.run(
        loop=loop,
        record=record,
        message=TEST_MESSAGE,
        capability=CAPABILITY,
    )


outcome = asyncio.run(_run())
print("state:", outcome.task_result.state)
print("answer:", outcome.task_result.answer)
print("run_id:", outcome.task_result.run_id)
print("checks:", outcome.checks)
print("passed:", outcome.passed)

### Step 5 — Inspect trace

In [ ]:
run_id = outcome.task_result.run_id or outcome.task_result.task_id
summary = session.summarize_trace(run_id)
print(summary)

if loop.trace_emitter is not None:
    for evt in loop.trace_emitter.events:
        print(evt.step, evt.message, evt.tags)

### Step 6 — Decide

Set `DECISION` to `keep`, `improve`, `pause`, or `delete` after reviewing output and trace.

In [ ]:
DECISION = ExperimentDecision.KEEP  # keep | improve | pause | delete
NOTES = ""

final_record = session.decide(record.experiment_id, DECISION, notes=NOTES)
print(f"decision: {final_record.decision.value}")
print(f"linked runs: {final_record.run_ids}")